# Spectroscopic reference set (part 2)

Real S-PLUS galaxies whose SFH labels come from **Bagpipes fits to their SDSS spectra under exactly the training prior** (`synthetic_model.py`), so they test the ML model on real data with labels it can match. Firefly (SDSS DR16) is an independent check of those fits, never a label.

**Inputs:** `dr3_corrected.csv`, `../Data/sdss_eboss_firefly-dr16.fits`, `../dustMaps/csfd/`, `training_set.parquet` (part 1), SDSS spectra (downloaded), S-PLUS Cloud aperture photometry (needs `SPLUS_USERNAME` / `SPLUS_PASSWORD`).
**Outputs:** `spectra/`, `spectral_fits/` (one `.h5` per fit), `spectral_fits/selection.csv` (selected galaxies, append-only), `spectral_fits/failures.csv`, `spectroscopic_reference.csv`, `firefly_comparison.png`.
**Run:** top to bottom with the *Inference4ETGs (MultiNest)* kernel, after `training_set.ipynb`. Budget-limited (`BUDGET_HOURS` = 3 h, about 3 waves of ~55 min): the fit stage fits as many of the selected galaxies as the budget allows (every grid cell stays represented), the rest is left for a later pass. Re-running skips finished fits, so raising `BUDGET_HOURS` later only fits the missing galaxies.

Spectra fit 3″ (SDSS) or 2″ (BOSS) fibres, so photometry is the 3″ S-PLUS aperture, and BOSS galaxies are reported separately. The **test** half of the split is used once, for the final ML evaluation; with a few tens of galaxies this is a pilot-size set.

In [ ]:
import math
import os
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy.io import fits
from astroquery.sdss import SDSS
from dustmaps.csfd import CSFDQuery
import joblib
from joblib import Parallel, delayed

import synthetic_model as sm

# Files and folders
SPLUS_FILE = "dr3_corrected.csv"
FIREFLY_FILE = "../Data/sdss_eboss_firefly-dr16.fits"
DUST_MAP_FILE = "../dustMaps/csfd/csfd_ebv.fits"
DUST_MASK_FILE = "../dustMaps/csfd/mask.fits"
TRAINING_FILE = "training_set.parquet"
SPECTRA_DIRECTORY = "spectra"
FITS_DIRECTORY = "spectral_fits"
PILOT_DIRECTORY = "spectral_fits/pilot"
FAILURES_FILE = "spectral_fits/failures.csv"
SELECTION_FILE = "spectral_fits/selection.csv"   # selected specobjids, append-only across passes
MOCK_FILE = "spectral_fits/mocks.joblib"   # mock spectra and true labels, kept so cached mock fits stay checkable
OUTPUT_FILE = "spectroscopic_reference.csv"
FIGURE_FILE = "firefly_comparison.png"
CREDENTIALS_FILE = "../.env"     # git-ignored; holds SPLUS_USERNAME and SPLUS_PASSWORD
APERTURE_CACHE_FILE = "splus_aperture_photometry.csv"   # raw 3" photometry of every S-PLUS x Firefly match

# Sample
MIN_PROB_GALAXY = 0.8
MATCH_RADIUS = 1.5               # [arcsec]
N_REDSHIFT_BINS = 4
N_AGE_BINS = 3
GALAXIES_PER_CELL = 5
N_REAL_GALAXIES = N_REDSHIFT_BINS * N_AGE_BINS * GALAXIES_PER_CELL   # 60
FIBER_DIAMETER = {"26": 3.0, "v5_13_0": 2.0}   # [arcsec] SDSS legacy and BOSS
SQL_CHUNK = 200                  # specobjids per SDSS SQL query

# Fits
N_WORKERS = 11
N_MOCKS = 6                      # 2 per SFH branch
MOCK_SFH_DRAWS = [0.25, 0.25, 0.75, 0.75, 0.99, 0.99]   # sfh_type draws: Dirichlet, skew-normal, + burst
MIN_MOCKS_RECOVERED = 5
MOCK_CHECK_LABELS = ["mw_age", "Z_fin", "f_old"]
FIRST_WAVE_REAL_GALAXIES = N_WORKERS - N_MOCKS            # 5 real galaxies share the first wave
BUDGET_HOURS = 3.0
# A ~25-min cap under load is ~4,000 iterations, too few for MultiNest to converge in 29 dimensions with 300 live
# points; unconverged fits are excluded from the split, so fewer converged fits are preferred
TARGET_MINUTES_PER_FIT = 55.0
ITERATION_SAFETY = 0.8           # MultiNest slows down as it converges
PILOT_ITERATIONS = 1000
SEED = 0

# S-PLUS aperture photometry. DR4 keeps one table per band, each with its own RA/Dec columns
SPLUS_RA_COLUMN = "RA"
SPLUS_DEC_COLUMN = "DEC"
APERTURE_SOURCES = {}
for band in sm.BANDS:
    APERTURE_SOURCES[band] = (f"dr4_dual.dr4_dual_{band.lower()}", f"{band}_aper_3", f"e_{band}_aper_3")
SPLUS_QUERY_TIMEOUT = 600        # [s]
SPLUS_EXECUTION_MODE = "sync"    # small queries; async jobs are capped at 4 concurrent per account
SPLUS_QUERY_ATTEMPTS = 3         # tries per S-PLUS query before giving up (count)
SPLUS_RETRY_WAIT = 30            # [s] wait after the first failed try; doubles after each further one

SSFR_PASSIVE = -11.0             # [log yr^-1] star-forming above, passive below

os.makedirs(SPECTRA_DIRECTORY, exist_ok=True)
os.makedirs(PILOT_DIRECTORY, exist_ok=True)
failures = []

## 1. Sample
S-PLUS DR3 galaxies matched to Firefly DR16 spectra within 1.5″, with spectroscopic redshift inside the prior. One spectrum per S-PLUS galaxy (the closest match).

In [ ]:
# 1. S-PLUS galaxies
splus = pd.read_csv(SPLUS_FILE)
splus = splus[splus["PROB_GAL"] > MIN_PROB_GALAXY].reset_index(drop=True)

# 2. Firefly galaxies with a redshift inside the prior, in the S-PLUS declination range
FIREFLY_COLUMNS = ["SPECOBJID", "RUN2D", "PLATE", "MJD", "FIBERID", "RA", "DEC", "Z_NOQSO", "CLASS_NOQSO",
                   "Chabrier_MILES_age_massW", "Chabrier_MILES_metallicity_massW", "Chabrier_MILES_total_mass"]
firefly_data = fits.getdata(FIREFLY_FILE, 1)
firefly = {}
for column in FIREFLY_COLUMNS:
    values = np.asarray(firefly_data[column])
    # FITS stores big-endian numbers; pandas needs the machine's native byte order
    firefly[column] = values.astype(values.dtype.newbyteorder("="))
firefly = pd.DataFrame(firefly)
firefly["CLASS_NOQSO"] = firefly["CLASS_NOQSO"].str.strip()
firefly["RUN2D"] = firefly["RUN2D"].str.strip()
is_galaxy = firefly["CLASS_NOQSO"] == "GALAXY"
in_prior = (firefly["Z_NOQSO"] > sm.Z_RANGE[0]) & (firefly["Z_NOQSO"] < sm.Z_RANGE[1])
in_footprint = (firefly["DEC"] > splus["DEC"].min() - 1) & (firefly["DEC"] < splus["DEC"].max() + 1)
firefly = firefly[is_galaxy & in_prior & in_footprint].reset_index(drop=True)
del firefly_data

# 3. Nearest S-PLUS galaxy of each spectrum
spectrum_coordinates = SkyCoord(firefly["RA"].to_numpy() * u.deg, firefly["DEC"].to_numpy() * u.deg)
splus_coordinates = SkyCoord(splus["RA"].to_numpy() * u.deg, splus["DEC"].to_numpy() * u.deg)
nearest_splus, separation, _ = match_coordinates_sky(spectrum_coordinates, splus_coordinates)
firefly["splus_index"] = nearest_splus
firefly["separation"] = separation.arcsec
matched = firefly[firefly["separation"] < MATCH_RADIUS]

# 4. One spectrum per S-PLUS galaxy: the closest
matched = matched.sort_values("separation").drop_duplicates("splus_index").reset_index(drop=True)
# Kept as text: eBOSS specobjids exceed the int64 range
matched["specobjid"] = matched["SPECOBJID"].str.strip()
matched["fiber_diameter"] = matched["RUN2D"].map(FIBER_DIAMETER)
for column in ("Chabrier_MILES_age_massW", "Chabrier_MILES_metallicity_massW", "Chabrier_MILES_total_mass"):
    missing = matched[column] <= sm.FIREFLY_MISSING
    matched.loc[missing, column] = np.nan
print(f"{len(matched)} S-PLUS galaxies with an SDSS/BOSS spectrum; "
      f"{matched['fiber_diameter'].eq(3.0).sum()} SDSS, {matched['fiber_diameter'].eq(2.0).sum()} BOSS")

In [ ]:
# 1. Median S/N of each spectrum from the SDSS database
sn_by_id = {}
identifiers = matched["specobjid"].tolist()
for start in range(0, len(identifiers), SQL_CHUNK):
    chunk = identifiers[start:start + SQL_CHUNK]
    id_list = ", ".join(chunk)
    query = ("SELECT CAST(specObjID AS VARCHAR(20)) AS specobjid_text, snMedian "
             f"FROM SpecObjAll WHERE specObjID IN ({id_list})")
    result = SDSS.query_sql(query, data_release=16)
    for row in result:
        sn_by_id[str(row["specobjid_text"]).strip()] = float(row["snMedian"])
matched["sn_median"] = matched["specobjid"].map(sn_by_id)

# 2. Grid of redshift x Firefly age quantile bins; highest S/N first in each cell
candidates = matched.dropna(subset=["Chabrier_MILES_age_massW", "sn_median"]).copy()
candidates["redshift_bin"] = pd.qcut(candidates["Z_NOQSO"], N_REDSHIFT_BINS, labels=False)
candidates["age_bin"] = pd.qcut(candidates["Chabrier_MILES_age_massW"], N_AGE_BINS, labels=False)

chosen = []
for (redshift_bin, age_bin), cell in candidates.groupby(["redshift_bin", "age_bin"]):
    best = cell.sort_values("sn_median", ascending=False).head(GALAXIES_PER_CELL).copy()
    best["rank_in_cell"] = np.arange(len(best))
    chosen.append(best)
# Round-robin order (best of every cell first), so a budget cut never empties a whole cell
picks = pd.concat(chosen, ignore_index=True)
picks = picks.sort_values(["rank_in_cell", "redshift_bin", "age_bin"]).reset_index(drop=True)
print(f"{len(picks)} galaxies picked this pass (target {N_REAL_GALAXIES})")
display(pd.crosstab(picks["redshift_bin"], picks["age_bin"]))

# 3. Selection kept across passes. The stored ids stay, in their stored order (so the first wave and the
# split never change), and this pass's new picks are appended: a later pass can only add galaxies
stored_identifiers = []
if os.path.exists(SELECTION_FILE):
    stored_identifiers = pd.read_csv(SELECTION_FILE, dtype=str)["specobjid"].tolist()
selection_identifiers = list(stored_identifiers)
for specobjid in picks["specobjid"]:
    if specobjid not in selection_identifiers:
        selection_identifiers.append(specobjid)
pd.DataFrame({"specobjid": selection_identifiers}).to_csv(SELECTION_FILE, index=False)

# 4. Reference galaxies: the stored selection plus every galaxy that already has a fit file (mock and pilot
# fits are not galaxies), so a pass that changes the picks can never drop a fitted or split galaxy
fitted_identifiers = []
for file_name in sorted(os.listdir(FITS_DIRECTORY)):
    if file_name.endswith(".h5") and not file_name.startswith(("mock_", "pilot")):
        fitted_identifiers.append(file_name[:-len(".h5")])
reference_identifiers = list(selection_identifiers)
for specobjid in fitted_identifiers:
    if specobjid not in reference_identifiers:
        reference_identifiers.append(specobjid)

available = set(matched["specobjid"])
lost_identifiers = [specobjid for specobjid in reference_identifiers if specobjid not in available]
if lost_identifiers:
    print(f"WARNING: {len(lost_identifiers)} stored galaxies are no longer in the S-PLUS x Firefly match "
          f"and cannot be rebuilt: {lost_identifiers}")
reference_identifiers = [specobjid for specobjid in reference_identifiers if specobjid in available]
selected = matched.set_index("specobjid").loc[reference_identifiers].reset_index()
selected = selected.rename(columns={"RA": "ra", "DEC": "dec", "Z_NOQSO": "z", "PLATE": "plate", "MJD": "mjd",
                                    "FIBERID": "fiberid", "Chabrier_MILES_age_massW": "firefly_age",
                                    "Chabrier_MILES_metallicity_massW": "firefly_metallicity",
                                    "Chabrier_MILES_total_mass": "firefly_mass"})
print(f"{len(selected)} reference galaxies: {len(stored_identifiers)} stored, "
      f"{len(selection_identifiers) - len(stored_identifiers)} newly picked, "
      f"{len(fitted_identifiers)} with a fit file")

## 2. Spectra
SDSS spectra are **not** corrected for Galactic extinction, so each is dereddened with the same map and law as the S-PLUS photometry (CSFD, CCM89, $R_V = 3.1$), converted to air wavelengths (the models' frame), masked (bad pixels, 5577/6300 Å sky lines) and rebinned to 5 Å.

In [ ]:
dust_map = CSFDQuery(map_fname=DUST_MAP_FILE, mask_fname=DUST_MASK_FILE)
galaxy_coordinates = SkyCoord(selected["ra"].to_numpy() * u.deg, selected["dec"].to_numpy() * u.deg)
# The map is a big-endian FITS image; float64 is native, which pandas needs
selected["ebv"] = dust_map(galaxy_coordinates).astype(float)

prepared_spectra = {}
for row in selected.itertuples():
    spectrum_file = os.path.join(SPECTRA_DIRECTORY, f"{row.specobjid}.fits")

    # 1. Download once
    if not os.path.exists(spectrum_file):
        try:
            downloaded = SDSS.get_spectra(plate=int(row.plate), mjd=int(row.mjd), fiberID=int(row.fiberid),
                                          data_release=16)
            downloaded[0].writeto(spectrum_file)
        except Exception as error:
            failures.append({"job_id": str(row.specobjid), "stage": "download", "error": repr(error)})
            continue

    # 2. Prepare for the fit
    with fits.open(spectrum_file) as spectrum_hdus:
        table = spectrum_hdus[1].data
        wavelength_vacuum = 10 ** table["loglam"].astype(float)
        flux = table["flux"].astype(float)
        ivar = table["ivar"].astype(float)
    try:
        prepared_spectra[row.specobjid] = sm.prepare_spectrum(wavelength_vacuum, flux, ivar, row.ebv)
    except ValueError as error:
        failures.append({"job_id": str(row.specobjid), "stage": "prepare", "error": repr(error)})

print(f"{len(prepared_spectra)} spectra ready; {len(failures)} failures so far")

## 3. Aperture photometry
3″-diameter S-PLUS magnitudes, matching the SDSS fibre, dereddened like `dr3_corrected.csv`. They set the mass normalisation of the labels and feed the coverage test; they never enter the spectral fit.

The raw (not dereddened) magnitudes of every S-PLUS x Firefly match are cached in `splus_aperture_photometry.csv`; the S-PLUS Cloud is queried only for selected galaxies missing from that cache (each query is retried up to `SPLUS_QUERY_ATTEMPTS` times), and every newly queried galaxy is appended to the cache, so a re-run needs no live queries.

In [ ]:
# Output column names, built once
aperture_columns = []
aperture_error_columns = []
for band in sm.BANDS:
    aperture_columns.append(f"{band}_aper")
    aperture_error_columns.append(f"e_{band}_aper")

# Columns of the cache file, in file order
cache_columns = ["specobjid", "ra", "dec", "splus_ra", "splus_dec", "match_separation_arcsec"]
for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
    cache_columns.append(magnitude_column)
    cache_columns.append(error_column)


def query_to_dataframe(result):
    """S-PLUS Cloud query result as a DataFrame.

    Inputs: result: what splusdata's Core.query returns, an astropy Table or a DataFrame.
    Returns: pandas DataFrame with the same rows and columns.
    """
    if hasattr(result, "to_pandas"):
        return result.to_pandas()
    return pd.DataFrame(result)


def cone_query(table, columns, ra, dec, radius_arcsec):
    """ADQL selecting columns of every object within radius of one position.

    One position per query: the S-PLUS Cloud rejects several OR-ed cones with "Column 'ra' does not exist".
    Inputs: table: S-PLUS Cloud table name (str); columns: names of the columns to select besides the
            position (list of str); ra, dec: cone centre [deg]; radius_arcsec: cone radius [arcsec].
    Returns: the ADQL query text (str).
    """
    radius_degrees = radius_arcsec / 3600
    column_list = ", ".join([SPLUS_RA_COLUMN, SPLUS_DEC_COLUMN, *columns])
    return (f"SELECT {column_list} FROM {table} WHERE 1 = CONTAINS(POINT('ICRS', {SPLUS_RA_COLUMN}, "
            f"{SPLUS_DEC_COLUMN}), CIRCLE('ICRS', {ra}, {dec}, {radius_degrees}))")


def run_query_with_retries(connection, query):
    """Run one S-PLUS Cloud query, trying again after a failure (the server is slow and sometimes drops requests).

    Waits SPLUS_RETRY_WAIT seconds after the first failure and twice as long after each further one.
    Inputs: connection: logged-in splusdata.Core; query: ADQL text (str).
    Returns: DataFrame with the query result (empty if no object was found).
    Raises: the last error if all SPLUS_QUERY_ATTEMPTS tries fail.
    """
    wait = SPLUS_RETRY_WAIT
    for attempt in range(1, SPLUS_QUERY_ATTEMPTS + 1):
        try:
            return query_to_dataframe(connection.query(query, execution_mode=SPLUS_EXECUTION_MODE,
                                                       timeout=SPLUS_QUERY_TIMEOUT))
        except Exception as error:
            if attempt == SPLUS_QUERY_ATTEMPTS:
                raise
            print(f"S-PLUS query failed (try {attempt} of {SPLUS_QUERY_ATTEMPTS}): {error!r}; "
                  f"trying again in {wait} s")
            time.sleep(wait)
            wait = 2 * wait


def query_one_galaxy(connection, specobjid, ra, dec):
    """Raw 3" aperture photometry of one galaxy, as one row of the cache file.

    Each band is a separate table, so there is one query per band; in each, the closest object within
    MATCH_RADIUS is taken. splus_ra, splus_dec and match_separation_arcsec describe the closest object of
    the first band in which one was found.
    Inputs: connection: logged-in splusdata.Core; specobjid: SDSS identifier (str); ra, dec: SDSS position [deg].
    Returns: dict with the cache_columns (magnitudes [mag] and errors [mag] are NaN where nothing was found).
    """
    row = dict.fromkeys(cache_columns, np.nan)
    row.update(specobjid=specobjid, ra=ra, dec=dec)
    galaxy_position = SkyCoord(ra * u.deg, dec * u.deg)
    for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
        query = cone_query(table, [magnitude_column, error_column], ra, dec, MATCH_RADIUS)
        result = run_query_with_retries(connection, query)
        if len(result) == 0:
            continue
        result_positions = SkyCoord(result[SPLUS_RA_COLUMN].to_numpy() * u.deg,
                                    result[SPLUS_DEC_COLUMN].to_numpy() * u.deg)
        separations = result_positions.separation(galaxy_position).arcsec
        closest = np.argmin(separations)
        row[magnitude_column] = result[magnitude_column].iloc[closest]
        row[error_column] = result[error_column].iloc[closest]
        if np.isnan(row["splus_ra"]):
            row["splus_ra"] = result[SPLUS_RA_COLUMN].iloc[closest]
            row["splus_dec"] = result[SPLUS_DEC_COLUMN].iloc[closest]
            row["match_separation_arcsec"] = separations[closest]
    return row


def query_aperture_photometry(galaxies):
    """Raw 3" aperture magnitudes and errors of galaxies from the S-PLUS Cloud, saved to the cache as they arrive.

    Each galaxy is appended to APERTURE_CACHE_FILE as soon as it is queried (a galaxy with no S-PLUS object
    is stored as NaN), so an interrupted run loses nothing.
    Inputs: galaxies: DataFrame with columns specobjid (str), ra and dec [deg].
    Returns: DataFrame indexed like galaxies, with {band}_aper and e_{band}_aper columns [mag] (NaN if not found).
    """
    import splusdata
    from dotenv import load_dotenv

    load_dotenv(CREDENTIALS_FILE)
    connection = splusdata.Core(os.environ["SPLUS_USERNAME"], os.environ["SPLUS_PASSWORD"])

    photometry = pd.DataFrame(np.nan, index=galaxies.index, columns=aperture_columns + aperture_error_columns)
    for galaxy in galaxies.itertuples():
        row = query_one_galaxy(connection, galaxy.specobjid, galaxy.ra, galaxy.dec)
        write_header = not os.path.exists(APERTURE_CACHE_FILE)
        pd.DataFrame([row], columns=cache_columns).to_csv(APERTURE_CACHE_FILE, mode="a", header=write_header,
                                                          index=False)
        for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
            photometry.loc[galaxy.Index, f"{band}_aper"] = row[magnitude_column]
            photometry.loc[galaxy.Index, f"e_{band}_aper"] = row[error_column]
    return photometry


# 1. Raw magnitudes from the cache, by specobjid
for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
    selected[f"{band}_aper"] = np.nan
    selected[f"e_{band}_aper"] = np.nan
in_cache = pd.Series(False, index=selected.index)
if os.path.exists(APERTURE_CACHE_FILE):
    cache = pd.read_csv(APERTURE_CACHE_FILE, dtype={"specobjid": str})
    cache = cache.drop_duplicates("specobjid").set_index("specobjid")
    in_cache = selected["specobjid"].isin(cache.index)
    cached = cache.loc[selected.loc[in_cache, "specobjid"]]
    for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
        selected.loc[in_cache, f"{band}_aper"] = cached[magnitude_column].to_numpy()
        selected.loc[in_cache, f"e_{band}_aper"] = cached[error_column].to_numpy()
print(f"{in_cache.sum()} of {len(selected)} galaxies found in {APERTURE_CACHE_FILE}")

# 2. The rest from the S-PLUS Cloud (appended to the cache as they arrive)
if not in_cache.all():
    queried = query_aperture_photometry(selected[~in_cache])
    selected.loc[queried.index, queried.columns] = queried

# 3. Galactic extinction, as for dr3_corrected.csv (non-detections keep their 99)
raw_magnitudes = selected[aperture_columns].to_numpy()
dereddened = sm.deredden_magnitudes(raw_magnitudes, selected["ebv"].to_numpy())
measured = raw_magnitudes < sm.UNDETECTED_MAGNITUDE
selected[aperture_columns] = np.where(measured, dereddened, raw_magnitudes)

missing_photometry = selected["r_aper"].isna()
print(f"Aperture photometry found for {(~missing_photometry).sum()} of {len(selected)} galaxies")
assert (~missing_photometry).sum() >= 0.9 * len(selected), "Aperture photometry missing for >10% of galaxies"

## 4. Fits
**Pilot:** `N_WORKERS` short fits run in parallel, on different galaxies, measure the time per MultiNest iteration under the same load as the real fits; it sets the iteration cap for ~55 min fits. The budget allows about 3 waves of fits within `BUDGET_HOURS`, so only part of the selected galaxies is fitted (round-robin order keeps every grid cell represented); `GALAXIES_PER_CELL` stays 5 so a later pass can fit the rest. **Inject-and-recover:** 6 mocks (2 per SFH branch) with real wavelengths, masks and errors run in the first wave with 5 real galaxies; at least 5 must recover the true `mw_age`, `Z_fin` and `f_old` inside their 95% intervals before the remaining fits start. Fits that hit the cap are flagged `converged = False`. If the check fails, the first-wave real fits and the mock fits are moved to `spectral_fits/failed_validation_<k>/` before the error is raised, so a re-run starts clean. Each wave prints its real duration against `TARGET_MINUTES_PER_FIT` (wall time inside a wave is not bounded).

In [ ]:
rng = np.random.default_rng(SEED)

# 1. Pilot under load: one short fit per worker, all at once, on different galaxies
pilot_jobs = []
for pilot_galaxy in list(prepared_spectra)[:N_WORKERS]:
    pilot_redshift = float(selected.loc[selected["specobjid"] == pilot_galaxy, "z"].iloc[0])
    # The iteration count is in the job id: a pilot file from a different PILOT_ITERATIONS is never reused
    pilot_jobs.append({"job_id": f"pilot_{PILOT_ITERATIONS}_{pilot_galaxy}", "spectrum": prepared_spectra[pilot_galaxy],
                       "redshift": pilot_redshift})
pilot_summaries = Parallel(n_jobs=N_WORKERS, verbose=5)(
    delayed(sm.fit_and_save)(job, PILOT_DIRECTORY, PILOT_ITERATIONS) for job in pilot_jobs
)

# 2. Seconds per iteration: mean over the pilots that ran
pilot_speeds = []
for pilot in pilot_summaries:
    if pilot["status"] in ("fitted", "skipped"):
        pilot_speeds.append(pilot["run_time"] / pilot["n_iterations"])
    else:
        print(f"Pilot {pilot['job_id']} failed: {pilot['error']}")
assert len(pilot_speeds) > 0, "No pilot fit ran; investigate first"
seconds_per_iteration = float(np.mean(pilot_speeds))

# 3. Iteration cap and projected time of all the fits
max_iterations = int(ITERATION_SAFETY * TARGET_MINUTES_PER_FIT * 60 / seconds_per_iteration)
n_fits = N_MOCKS + len(prepared_spectra)
n_waves = math.ceil(n_fits / N_WORKERS)
projected_hours = n_waves * TARGET_MINUTES_PER_FIT / 60
print(f"Pilot ({len(pilot_speeds)} parallel fits): {seconds_per_iteration:.2f} s per iteration -> "
      f"cap of {max_iterations} iterations per fit")
print(f"Projected: {n_fits} fits in {n_waves} waves of ~{TARGET_MINUTES_PER_FIT:.0f} min "
      f"= {projected_hours:.1f} h (budget {BUDGET_HOURS} h)")
if projected_hours > BUDGET_HOURS:
    print("WARNING: the projection exceeds the budget; cell 12 will cut galaxies after the first wave "
          "(round-robin order keeps every grid cell represented). Raise BUDGET_HOURS and re-run to fit the rest.")

In [ ]:
def run_wave(jobs):
    """Fit a list of jobs on N_WORKERS processes, then print how long the wave took against the target.

    Inputs: jobs: list of dicts with job_id (str), spectrum (dict as from prepare_spectrum) and
            redshift (dimensionless); the iteration limit is the global max_iterations (count).
    Returns: list of summary dicts from sm.fit_and_save (job_id, status, converged, run_time [s], ...).
    """
    wave_start = time.time()
    summaries = Parallel(n_jobs=N_WORKERS, verbose=5)(
        delayed(sm.fit_and_save)(job, FITS_DIRECTORY, max_iterations) for job in jobs
    )
    wave_minutes = (time.time() - wave_start) / 60
    print(f"Wave of {len(jobs)} fits took {wave_minutes:.0f} min "
          f"(target {TARGET_MINUTES_PER_FIT:.0f} min, {wave_minutes - TARGET_MINUTES_PER_FIT:+.0f} min)")
    return summaries


def real_galaxy_job(specobjid):
    """Fit job of one selected galaxy.

    Inputs: specobjid: SDSS identifier (str) of a galaxy in selected with a prepared spectrum.
    Returns: dict with job_id (str), spectrum (dict from prepare_spectrum: wavelength [Angstrom], flux, ivar)
             and redshift (spectroscopic, dimensionless).
    """
    redshift = float(selected.loc[selected["specobjid"] == specobjid, "z"].iloc[0])
    return {"job_id": str(specobjid), "spectrum": prepared_spectra[specobjid], "redshift": redshift}


def archive_failed_validation(job_ids):
    """Move the fits of the given jobs and the stored mocks into spectral_fits/failed_validation_<k>/.

    For each job the .h5 file and the MultiNest folder move; MOCK_FILE moves too, so the next run draws
    new mocks and refits everything that was moved. k is the first integer whose folder does not exist yet.
    Inputs: job_ids: identifiers (list of str) of the fits to archive.
    Returns: path of the archive folder (str).
    """
    k = 1
    while os.path.exists(os.path.join(FITS_DIRECTORY, f"failed_validation_{k}")):
        k += 1
    archive_folder = os.path.join(FITS_DIRECTORY, f"failed_validation_{k}")
    os.makedirs(archive_folder)
    for job_id in job_ids:
        for name in (f"{job_id}.h5", str(job_id)):
            source = os.path.join(FITS_DIRECTORY, name)
            if os.path.exists(source):
                shutil.move(source, archive_folder)
    if os.path.exists(MOCK_FILE):
        shutil.move(MOCK_FILE, archive_folder)
    return archive_folder


# 1. Mock jobs: real wavelengths, masks and errors; known unit-cube points. The mocks are stored with their
# true labels: the mock fits are cached on disk, so a re-run (even with more galaxies) must reuse the same
# mocks, not draw new ones that the stored posteriors no longer match
real_identifiers = list(prepared_spectra)
if os.path.exists(MOCK_FILE):
    mock_jobs, mock_truths = joblib.load(MOCK_FILE)
    print(f"Reusing {len(mock_jobs)} stored mocks from {MOCK_FILE}")
else:
    mock_jobs = []
    mock_truths = {}
    sfh_type_index = sm.SPECTRAL_DIMS.index("sfh_type")
    for mock_index, sfh_type_draw in enumerate(MOCK_SFH_DRAWS):
        template_identifier = real_identifiers[rng.integers(len(real_identifiers))]
        template_job = real_galaxy_job(template_identifier)
        cube = rng.random(len(sm.SPECTRAL_DIMS))
        cube[sfh_type_index] = sfh_type_draw
        mock_spectrum, truth = sm.make_mock_spectrum(cube, template_job["redshift"], template_job["spectrum"], rng)
        job_id = f"mock_{mock_index}"
        mock_jobs.append({"job_id": job_id, "spectrum": mock_spectrum, "redshift": template_job["redshift"]})
        mock_truths[job_id] = truth
        # A mock fit without stored truths cannot be checked: remove it so it is fitted again
        stale_fit = os.path.join(FITS_DIRECTORY, f"{job_id}.h5")
        if os.path.exists(stale_fit):
            os.remove(stale_fit)
    joblib.dump((mock_jobs, mock_truths), MOCK_FILE)

# 2. First wave
fitting_start = time.time()
first_wave_jobs = list(mock_jobs)
for specobjid in real_identifiers[:FIRST_WAVE_REAL_GALAXIES]:
    first_wave_jobs.append(real_galaxy_job(specobjid))
fit_summaries = run_wave(first_wave_jobs)
first_wave_hours = (time.time() - fitting_start) / 3600

# 3. Inject-and-recover: truth inside the 95% interval for every checked label
recovered = 0
for job in mock_jobs:
    mock_file = os.path.join(FITS_DIRECTORY, f"{job['job_id']}.h5")
    if not os.path.exists(mock_file):
        print(f"{job['job_id']}: no fit file (the fit failed)")
        continue
    posterior, _ = sm.load_fit(mock_file)
    inside_all = True
    for label in MOCK_CHECK_LABELS:
        low, high = np.nanpercentile(posterior[label], [2.5, 97.5])
        truth_value = mock_truths[job["job_id"]][label]
        inside = low <= truth_value <= high
        inside_all = inside_all and inside
        print(f"{job['job_id']} {label}: truth {truth_value:.3f}, 95% [{low:.3f}, {high:.3f}] {'ok' if inside else 'MISS'}")
    recovered += int(inside_all)
print(f"{recovered} of {len(mock_jobs)} mocks recovered; first wave took {first_wave_hours:.2f} h")

# A failed check must not leave fits behind that a re-run would trust: archive them before raising
if recovered < MIN_MOCKS_RECOVERED:
    first_wave_ids = [job["job_id"] for job in first_wave_jobs]
    archive_folder = archive_failed_validation(first_wave_ids)
    raise AssertionError(f"Inject-and-recover failed ({recovered} of {len(mock_jobs)} mocks recovered, "
                         f"{MIN_MOCKS_RECOVERED} needed). The first-wave real fits and the mock fits were moved "
                         f"to {archive_folder}; investigate before re-running (new mocks are drawn then).")

In [ ]:
# Fit as many of the remaining galaxies as the budget allows, at the measured wave time
# Galaxies that already have a fit file need no budget; on a later pass the budget goes to the new ones
remaining = []
for specobjid in real_identifiers[FIRST_WAVE_REAL_GALAXIES:]:
    if not os.path.exists(os.path.join(FITS_DIRECTORY, f"{specobjid}.h5")):
        remaining.append(specobjid)
hours_left = BUDGET_HOURS - (time.time() - fitting_start) / 3600
# On a re-run the first wave is skipped and takes no time; fall back to the target fit time
if first_wave_hours < 0.05:
    wave_hours = TARGET_MINUTES_PER_FIT / 60
else:
    wave_hours = first_wave_hours
# A first wave that overran the budget leaves no waves; never a negative number (it would slice from the end)
waves_left = max(0, int(hours_left // wave_hours))
n_allowed = waves_left * N_WORKERS
if n_allowed < len(remaining):
    print(f"Budget allows {n_allowed} of the {len(remaining)} remaining galaxies; "
          f"fitting the first {n_allowed}. Raise BUDGET_HOURS and re-run to fit the rest.")
    remaining = remaining[:n_allowed]

remaining_jobs = []
for specobjid in remaining:
    remaining_jobs.append(real_galaxy_job(specobjid))
fit_summaries = fit_summaries + run_wave(remaining_jobs)

summary_table = pd.DataFrame(fit_summaries)
for record in fit_summaries:
    if record["status"] == "failed":
        failures.append({"job_id": record["job_id"], "stage": "fit", "error": record["error"]})
pd.DataFrame(failures).to_csv(FAILURES_FILE, index=False)
print(f"Fitting took {(time.time() - fitting_start) / 3600:.2f} h")
display(summary_table["status"].value_counts())
display(summary_table.groupby("status")[["run_time", "n_iterations"]].median())

## 5. Labels, Firefly check, coverage and split
Each posterior sample's mass-like labels are rescaled by its aperture $r$ offset ($-0.4\,\Delta m$ dex), then summarised as 16/50/84th percentiles. A fit is **Firefly-consistent** if mass-weighted age and metallicity agree within 0.3 dex; failures are flagged, not deleted. Coverage uses the aperture photometry at the spectroscopic redshift. The table has every galaxy of the stored selection (`selection.csv`) and every galaxy with a fit file, so a later pass never drops one. `converged` is recomputed from each fit's MultiNest folder (`sm.multinest_iterations` below the iteration limit), because the stored flag of older fits counted one ev.dat row per mode and iteration. Galaxies that converged, are covered and are Firefly-consistent go to `validation` or `test` by a hash of `specobjid`. Read the output with `dtype={"specobjid": str}` (eBOSS ids exceed int64).

In [ ]:
LABEL_COLUMNS = ["log_ssfr", "mw_metallicity", "veldisp", "noise_scale", "Z_fin", "Av_ism",
                 *sm.SUMMARY_LABEL_COLUMNS, *sm.BIN_LABEL_COLUMNS]

def converged_from_multinest(specobjid, metadata):
    """Whether a fit certainly stopped before its iteration limit, judged from its MultiNest folder.

    Uses the upper bound of the outer-loop iteration count (sm.multinest_iteration_bounds), so a capped
    run is never called converged. The limit is the one stored in the fit's metadata. A fit saved before
    the limit was stored (or without its MultiNest folder) keeps the flag stored when it ran: that flag
    came from the ev.dat row count, which overcounts iterations, so a stored True is already safe and a
    stored False is never upgraded against a limit that may differ from the one the fit used.
    Inputs: specobjid: SDSS identifier (str); metadata: dict from sm.load_fit.
    Returns: True if the run converged, else False (bool).
    """
    iteration_limit = metadata.get("max_iterations")
    multinest_basename = os.path.join(FITS_DIRECTORY, specobjid) + "/"
    if iteration_limit is None or not os.path.exists(multinest_basename + "ev.dat"):
        return bool(metadata["converged"])
    return sm.multinest_iteration_bounds(multinest_basename)[1] < iteration_limit


# 1. Posterior summaries per galaxy (selected holds the stored selection and every fitted galaxy, see cell 4)
failed_identifiers = {failure["job_id"] for failure in failures}   # stage fit, prepare or download
rows = []
for galaxy in selected.itertuples():
    fit_file = os.path.join(FITS_DIRECTORY, f"{galaxy.specobjid}.h5")
    row = {"specobjid": galaxy.specobjid}
    if not os.path.exists(fit_file):
        # Failed galaxies are told apart from those the budget left for a later pass
        row["fit_status"] = "failed" if galaxy.specobjid in failed_identifiers else "not_fitted"
        rows.append(row)
        continue
    posterior, metadata = sm.load_fit(fit_file)

    # Mass-like labels follow the aperture r magnitude: log X = log X_model - 0.4 (r_obs - r_model)
    offsets = galaxy.r_aper - posterior["model_r"]
    posterior = sm.rescale_mass_labels(posterior, offsets)
    posterior["log_ssfr"] = posterior["log_sfr"] - posterior["log_mstar"]   # normalisation-free

    row["fit_status"] = "fitted"
    row["converged"] = converged_from_multinest(galaxy.specobjid, metadata)
    row["fit_hours"] = metadata["run_time"] / 3600
    row.update(sm.summarise_posterior(posterior, LABEL_COLUMNS))
    rows.append(row)
reference = selected.merge(pd.DataFrame(rows), on="specobjid", how="left")

# 2. Firefly check; its missing values (-9999) become NaN before any logarithm, as in sm.compare_with_firefly
for column in ("firefly_age", "firefly_metallicity", "firefly_mass"):
    reference[column] = reference[column].where(reference[column] > sm.FIREFLY_MISSING)
firefly_check = sm.compare_with_firefly(
    reference["mw_age_median"].to_numpy(), reference["mw_metallicity_median"].to_numpy(),
    reference["firefly_age"].to_numpy(), reference["firefly_metallicity"].to_numpy(),
)
for column in firefly_check.columns:
    reference[column] = firefly_check[column].to_numpy()
reference["delta_log_mass"] = reference["log_mstar_median"] - np.log10(reference["firefly_mass"])

# 3. Coverage of the aperture photometry at the spectroscopic redshift
training = pd.read_parquet(TRAINING_FILE, columns=["redshift", *sm.BANDS])
aperture_magnitudes = reference[aperture_columns].to_numpy()
aperture_errors = reference[aperture_error_columns].to_numpy()
aperture_detected = sm.detected_bands(aperture_magnitudes, aperture_errors)
coverage = sm.coverage_test(aperture_magnitudes, aperture_errors, aperture_detected,
                            reference["z"].to_numpy(), np.full(len(reference), sm.DZ_MIN),
                            training[sm.BANDS].to_numpy(), training["redshift"].to_numpy())
for column in coverage.columns:
    reference[column] = coverage[column].to_numpy()

# 4. Locked split
usable = (reference["converged"].fillna(False).astype(bool)
          & reference["covered"].astype(bool)
          & reference["firefly_consistent"].fillna(False).astype(bool))
reference["split"] = "excluded"
for row_index in np.flatnonzero(usable.to_numpy()):
    reference.loc[row_index, "split"] = sm.split_by_hash(reference.loc[row_index, "specobjid"])

reference.to_csv(OUTPUT_FILE, index=False)
fitted = reference["fit_status"] == "fitted"
assert reference.loc[fitted, "mw_age_median"].notna().all(), "A fitted galaxy has no labels in the output"
print(f"{OUTPUT_FILE}: {len(reference)} galaxies, {fitted.sum()} fitted, "
      f"{reference['converged'].fillna(False).sum()} converged, {reference['covered'].sum()} covered, "
      f"{reference['firefly_consistent'].fillna(False).sum()} Firefly-consistent")
display(reference["split"].value_counts())

In [ ]:
reference["fibre"] = reference["fiber_diameter"].map({3.0: "SDSS 3\"", 2.0: "BOSS 2\""})
reference["galaxy_type"] = np.where(reference["log_ssfr_median"] > SSFR_PASSIVE, "star-forming", "passive")

for grouping in ("fibre", "galaxy_type"):
    grouped = reference[fitted].groupby(grouping)
    table = pd.DataFrame({
        "median Δlog age": grouped["delta_log_age"].median(),
        "scatter Δlog age": grouped["delta_log_age"].std(),
        "median Δlog Z": grouped["delta_log_metallicity"].median(),
        "scatter Δlog Z": grouped["delta_log_metallicity"].std(),
        "median Δlog M": grouped["delta_log_mass"].median(),
        "consistent": grouped["firefly_consistent"].mean(),
        "galaxies": grouped.size(),
    })
    display(table.round(3))

figure, axes = plt.subplots(1, 2, figsize=(12, 5.5))
comparisons = [
    (np.log10(reference["firefly_age"]), np.log10(reference["mw_age_median"] * sm.GYR_TO_YEARS), "log age [yr]"),
    (np.log10(reference["firefly_metallicity"]), np.log10(reference["mw_metallicity_median"]), r"log $Z/Z_\odot$"),
]
for axis, (firefly_values, bagpipes_values, label) in zip(axes, comparisons):
    for galaxy_type, marker in (("star-forming", "o"), ("passive", "s")):
        chosen = fitted & (reference["galaxy_type"] == galaxy_type)
        axis.scatter(firefly_values[chosen], bagpipes_values[chosen], marker=marker, label=galaxy_type)
    limits = [np.nanmin([firefly_values, bagpipes_values]), np.nanmax([firefly_values, bagpipes_values])]
    axis.plot(limits, limits, color="black", lw=1)
    axis.set(xlabel=f"Firefly {label}", ylabel=f"Bagpipes {label}")
    axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_FILE, dpi=200)
plt.show()

## Next pass
Raise `GALAXIES_PER_CELL` (and `BUDGET_HOURS`) and re-run: the selection in `selection.csv` only grows (new picks are appended, every stored or fitted galaxy stays in the table), finished fits are skipped, the stored mocks (`MOCK_FILE`) are reused so the inject-and-recover check stays valid, and the hash split never moves an existing galaxy. Delete `MOCK_FILE` to draw new mocks (their old fits are then removed and refitted). Add DR4 Stripe 82 galaxies by pointing `SPLUS_FILE` at a DR4 catalogue with the same columns.